In [14]:
import pandas as pd
import nest_asyncio
from vllm import LLM, SamplingParams

In [19]:
dados = pd.read_csv('../dados/chamados_com_predicoes.csv', index_col='id_chamado')

In [ ]:
nest_asyncio.apply()

llm = LLM(
    model="Qwen/Qwen3.5-9B",
    dtype="float16",                  
    gpu_memory_utilization=0.85,      
    max_model_len=1024,               
    max_num_seqs=64,                  
    enforce_eager=True                
)

sampling_params = SamplingParams(
    temperature=0.0,  # Resposta determinística
    max_tokens=25,  # Textos curtos de classificação não precisam de mais que isso
    stop=[
        "<|im_end|>",
        "<|endoftext|>",
    ], 
)

textos_para_classificar = dados["texto"].tolist()
prompts_formatados = []

for texto in textos_para_classificar:
    prompt = f"""<|im_start|>system
    Você é um classificador direto de segurança. Avalie se o texto envolve brigas, assaltos, choques elétricos ou atropelamentos.
    Responda APENAS com "Sim" ou "Não" logo após fechar a tag de pensamento.<|im_end|>
    <|im_start|>user
    Texto: "{texto}"<|im_end|>
    <|im_start|>assistant
    <think>"""
    prompts_formatados.append(prompt)

print(f"Iniciando a classificação em lote de {len(prompts_formatados)} textos...")
outputs = llm.generate(prompts_formatados, sampling_params)

# 7. Processando os resultados rapidamente
resultados = []
for output in outputs:
    resposta_limpa = output.outputs[0].text.split("</think>")[-1].strip()
    resultados.append(resposta_limpa)

print("Processamento concluído!")
print(f"Exemplo de resultado: {resultados[0]}")

INFO 07-07 18:52:24 [api_utils.py:273] non-default args: {'dtype': 'float16', 'max_model_len': 1024, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3.5-4B'}
INFO 07-07 18:52:25 [model.py:598] Resolved architecture: Qwen3_5ForConditionalGeneration
WARNING 07-07 18:52:25 [model.py:2063] Casting torch.bfloat16 to torch.float16.
INFO 07-07 18:52:25 [model.py:1725] Using max model len 1024
WARNING 07-07 18:52:25 [vllm.py:1062] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 07-07 18:52:25 [vllm.py:1110] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 07-07 18:52:25 [kernel.py:276] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 07-07 18

(EngineCore pid=32438) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=32438) INFO 07-07 18:52:35 [parallel_state.py:1588] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://192.168.1.25:49403 backend=nccl
(EngineCore pid=32438) INFO 07-07 18:52:35 [parallel_state.py:1923] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=32438) INFO 07-07 18:52:35 [topk_topp_sampler.py:55] Using FlashInfer for top-p & top-k sampling.


(EngineCore pid=32438) [transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=32438) INFO 07-07 18:52:46 [gpu_model_runner.py:5160] Starting to load model Qwen/Qwen3.5-4B...
(EngineCore pid=32438) INFO 07-07 18:52:46 [cuda.py:539] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=32438) INFO 07-07 18:52:46 [mm_encoder_attention.py:373] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore pid=32438) INFO 07-07 18:52:46 [qwen_gdn_linear_attn.py:228] Using Triton/FLA GDN prefill kernel (requested=auto, head_k_dim=128).
(EngineCore pid=32438) INFO 07-07 18:52:46 [cuda.py:480] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=32438) INFO 07-07 18:52:46 [flash_attn.py:670] Using FlashAttention version 2
(EngineCore pid=32438) INFO 07-07 18:52:47 [weight_utils.py:849] Filesystem type for checkpoints: ZFS. Checkpoint size: 8.68 GiB. Available RAM: 27.47 GiB.
(EngineCore pid=32438) INFO 07-07 18:52:47 [weight_utils.

...

(EngineCore pid=32438) INFO 07-07 18:52:49 [default_loader.py:430] Loading weights took 2.29 seconds
(EngineCore pid=32438) INFO 07-07 18:52:50 [gpu_model_runner.py:5255] Model loading took 8.61 GiB memory and 3.193178 seconds
(EngineCore pid=32438) INFO 07-07 18:52:50 [interface.py:773] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
(EngineCore pid=32438) INFO 07-07 18:52:50 [interface.py:797] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
(EngineCore pid=32438) INFO 07-07 18:52:50 [gpu_model_runner.py:6271] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=32438) INFO 07-07 18:53:05 [gpu_worker.py:508] Available KV cache memory: 2.95 GiB
(EngineCore pid=32438) INFO 07-07 18:53:05 [kv_cache_utils.py:2146] GPU KV cache size: 37,478 tokens
(EngineCore pid=32438) INFO 07-07 18:53:05 [k

Rendering prompts:   0%|          | 0/5000 [00:00<?, ?it/s]

(EngineCore pid=32438) WARNING 07-07 18:53:05 [jit_monitor.py:106] Triton kernel JIT compilation during inference: _zero_kv_blocks_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=32438) WARNING 07-07 18:53:05 [jit_monitor.py:106] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=32438) WARNING 07-07 18:53:05 [jit_monitor.py:106] Triton kernel JIT compilation during inference: _causal_conv1d_fwd_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=32438) WARNING 07-07 18:53:05 [jit_monitor.py:106] Triton kernel JIT compilation during inference: _fused_post_conv_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=32438) WARNING 07-07 18:53:05 [jit_monitor.py:106] Triton kernel JIT compilation durin

Processed prompts: 100%|██████████| 5000/5000 [02:57<00:00, 28.15it/s, est. speed input: 3185.35 toks/s, output: 141.56 toks/s]

Processamento concluído!
Exemplo de resultado: Não


In [35]:
resultados

['Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Sim',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',


In [36]:
from collections import Counter

# Passa a sua lista de resultados direto no Counter
distribuicao = Counter(resultados)

print("Distribuição dos resultados:")
for classe, quantidade in distribuicao.items():
    porcentagem = (quantidade / len(resultados)) * 100
    print(f"- {classe}: {quantidade} ({porcentagem:.2f}%)")

Distribuição dos resultados:
- Não: 4909 (98.18%)
- Sim: 84 (1.68%)
- Thinking Process:

1.  **Analyze the Request:**
    *   Role: Direct safety classifier.: 7 (0.14%)


In [33]:
del llm

(EngineCore pid=31539) INFO 07-07 18:52:17 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=31539) INFO 07-07 18:52:17 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=31539) INFO 07-07 18:52:17 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=31539) INFO 07-07 18:52:17 [core.py:1227] [shutdown] EngineCore: exiting busy loop


In [29]:
outputs[0].outputs[0].text

'<think>\nThinking Process:\n\n1.  **'